# Orchestrator (Multi-Location)

Runs the pipeline for **each location** defined in `locations.json`, then optionally merges all results into a single combined CSV.

**`locations.json` format:**
```json
[
    {"name": "times_square", "lat": 40.7589, "lon": -73.9851, "walk_minutes": 15.0, "walk_speed_m_min": 80.0},
    {"name": "central_park", "lat": 40.7829, "lon": -73.9654, "walk_minutes": 10.0, "walk_speed_m_min": 80.0}
]
```

**Per location, runs:**
```
01_identifiers.ipynb            → csv/01_identifiers.csv
02_y_target.ipynb               → csv/02_y_target.csv
03_morphological.ipynb          → csv/03_morphological.csv
04_synergistic_proximity.ipynb  → csv/04_synergistic_proximity.csv
05_socioeconomic.ipynb          → csv/05_socioeconomic.csv
── [OPTIONAL] ──────────────────────────────────────────
06_joker.ipynb                  → csv/06_joker.csv
```

**Requirements:** PLUTO CSV at `../ramy/NYC_pluto_25v4_csv/pluto_25v4.csv` (used by 03 and 05).

**Workflow:**
1. Add locations to `locations.json`
2. Set `INCLUDE_JOKER = True/False` in Cell 2
3. **Restart kernel → Run All Cells**
4. Outputs: one CSV per location + one combined CSV with all locations

In [ ]:
import papermill as pm
import pandas as pd
import pathlib, time, os, tempfile, json
from datetime import datetime

os.makedirs("csv", exist_ok=True)

RUN_ID = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
print(f"papermill {pm.__version__}")
print(f"Run ID: {RUN_ID}")

In [ ]:
# ── Configuration ─────────────────────────────────────

PIPELINE = [
    ("01_identifiers.ipynb",           "01 · Identifiers"),
    ("02_y_target.ipynb",              "02 · Y-Target"),
    ("03_morphological.ipynb",         "03 · Morphological"),
    ("04_synergistic_proximity.ipynb", "04 · Synergistic Proximity"),
    ("05_socioeconomic.ipynb",         "05 · Socioeconomic"),
]

RUN_MODULE = {nb: True for nb, _ in PIPELINE}

INCLUDE_JOKER = False   # Set to True to include 06_joker (median income) — slow
KERNEL_NAME   = "python3"

# ── Load locations ────────────────────────────────────
with open("locations.json", encoding="utf-8") as f:
    LOCATIONS = json.load(f)

print(f"Locations ({len(LOCATIONS)}):")
for loc in LOCATIONS:
    r = loc.get("walk_minutes", 15) * loc.get("walk_speed_m_min", 80)
    print(f"  - {loc['name']:<20s} ({loc['lat']}, {loc['lon']})  radius {r:.0f} m")

print(f"\nPipeline:")
for nb, desc in PIPELINE:
    status = "RUN" if RUN_MODULE[nb] else "SKIP"
    print(f"  [{status}]  {desc}")
joker_status = "RUN" if INCLUDE_JOKER else "SKIP"
print(f"  [{joker_status}]  06 · Joker")

In [ ]:
# ── Run pipeline for each location ────────────────────

FINAL_COLUMNS = [
    "osm_id", "name", "lat", "lon", "label", "distance_m",
    "lot_area_sqft", "highway_type", "lanes",
    "dist_bus_stop_m", "dist_hospital_m", "dist_school_m", "dist_park_m",
    "res_ratio", "com_ratio",
    "median_income",
]

CSV_GROUPS = [
    ("csv/01_identifiers.csv",           ["osm_id", "name", "lat", "lon", "distance_m", "label"]),
    ("csv/03_morphological.csv",         ["highway_type", "lanes", "lot_area_sqft"]),
    ("csv/04_synergistic_proximity.csv", ["dist_bus_stop_m", "dist_hospital_m", "dist_school_m", "dist_park_m"]),
    ("csv/05_socioeconomic.csv",         ["com_ratio", "res_ratio"]),
    ("csv/06_joker.csv",                 ["median_income"]),
]

INTERMEDIATES = [
    "csv/00_base_data.csv", "csv/01_identifiers.csv", "csv/02_y_target.csv",
    "csv/03_morphological.csv", "csv/04_synergistic_proximity.csv",
    "csv/05_socioeconomic.csv", "csv/06_joker.csv",
]

location_csvs = []

for loc_idx, loc in enumerate(LOCATIONS):
    name     = loc["name"]
    lat      = loc["lat"]
    lon      = loc["lon"]
    walk_min = loc.get("walk_minutes", 15.0)
    walk_spd = loc.get("walk_speed_m_min", 80.0)
    radius_m = walk_min * walk_spd
    net_dist = int(radius_m + 300)

    print(f"\n{'#'*60}")
    print(f"  LOCATION {loc_idx+1}/{len(LOCATIONS)}: {name}")
    print(f"  ({lat}, {lon})  radius {radius_m:.0f} m")
    print(f"{'#'*60}")

    # Parameters to inject via papermill
    nb_params = {
        "01_identifiers.ipynb": {
            "LAT": lat, "LON": lon,
            "WALK_MINUTES": walk_min, "WALK_SPEED_M_MIN": walk_spd,
            "RADIUS_M": radius_m,
        },
        "03_morphological.ipynb": {
            "ORIGIN_LAT": lat, "ORIGIN_LON": lon,
            "NETWORK_DIST": net_dist,
        },
    }

    # ── Run notebooks 01–05 ───────────────────────────
    results = []
    pipeline_ok = True

    for nb_path, description in PIPELINE:
        if not RUN_MODULE.get(nb_path, True):
            print(f"  SKIP  {description}")
            results.append((nb_path, "skipped", 0))
            continue
        if not pathlib.Path(nb_path).exists():
            raise FileNotFoundError(f"Notebook not found: {nb_path}")

        print(f"\n{'='*55}")
        print(f"  {description}")
        print(f"{'='*55}")

        tmp = pathlib.Path(tempfile.mktemp(suffix=".ipynb"))
        t0 = time.time()
        try:
            params = nb_params.get(nb_path, {})
            pm.execute_notebook(
                nb_path, str(tmp),
                kernel_name=KERNEL_NAME,
                parameters=params or None,
                progress_bar=False,
            )
            elapsed = time.time() - t0
            print(f"  Status : OK  ({elapsed:.1f} s)")
            results.append((nb_path, "ok", elapsed))
        except pm.exceptions.PapermillExecutionError as e:
            elapsed = time.time() - t0
            print(f"  Status : FAILED  ({elapsed:.1f} s)")
            print(f"  Error  : {e}")
            results.append((nb_path, "failed", elapsed))
            pipeline_ok = False
            break
        finally:
            if tmp.exists():
                tmp.unlink()

    # ── Optional: Joker ───────────────────────────────
    if pipeline_ok and INCLUDE_JOKER:
        print(f"\n{'='*55}")
        print(f"  06 · Joker (median income)")
        print(f"{'='*55}")
        tmp = pathlib.Path(tempfile.mktemp(suffix=".ipynb"))
        t0 = time.time()
        try:
            pm.execute_notebook(
                "06_joker.ipynb", str(tmp),
                kernel_name=KERNEL_NAME, progress_bar=False,
            )
            elapsed = time.time() - t0
            print(f"  Status : OK  ({elapsed:.1f} s)")
            results.append(("06_joker.ipynb", "ok", elapsed))
        except pm.exceptions.PapermillExecutionError as e:
            print(f"  Status : FAILED  ({elapsed:.1f} s) — {e}")
            results.append(("06_joker.ipynb", "failed", 0))
        finally:
            if tmp.exists():
                tmp.unlink()

    # ── Summary ───────────────────────────────────────
    print(f"\n  SUMMARY: {name}")
    for nb_path, status, elapsed in results:
        icon = {"ok": "OK", "failed": "FAIL", "skipped": "SKIP"}.get(status, "?")
        t_str = f"{elapsed:.1f} s" if elapsed else "-"
        print(f"  [{icon:4s}]  {nb_path:<42s} {t_str}")

    if not pipeline_ok:
        print(f"\n  Pipeline failed for {name} — skipping merge.")
        # Cleanup intermediates anyway
        for f in INTERMEDIATES:
            p = pathlib.Path(f)
            if p.exists():
                p.unlink()
        continue

    # ── Merge per location ────────────────────────────
    print(f"\nMerging outputs for {name}...")
    df_loc = pd.read_csv("csv/01_identifiers.csv")

    for csv_path, cols in CSV_GROUPS[1:]:
        if pathlib.Path(csv_path).exists():
            df_other = pd.read_csv(csv_path)
            df_loc = df_loc.merge(df_other, on="osm_id", how="left")
            print(f"  OK    {csv_path}")
        else:
            for col in cols:
                df_loc[col] = None
            print(f"  WARN  {csv_path} not found — columns filled with NaN")

    df_loc = df_loc[[c for c in FINAL_COLUMNS if c in df_loc.columns]]
    df_loc.insert(0, "location", name)

    loc_csv = f"csv/{name}_{RUN_ID}.csv"
    df_loc.to_csv(loc_csv, index=False, encoding="utf-8")
    location_csvs.append(loc_csv)
    print(f"  Saved: {loc_csv}  ({len(df_loc)} rows x {df_loc.shape[1]} cols)")

    # Cleanup intermediates
    for f in INTERMEDIATES:
        p = pathlib.Path(f)
        if p.exists():
            p.unlink()

print(f"\n{'#'*60}")
print(f"  ALL LOCATIONS COMPLETE")
print(f"  {len(location_csvs)}/{len(LOCATIONS)} succeeded")
print(f"{'#'*60}")

In [ ]:
# ── Combine all locations into one CSV ────────────────

if not location_csvs:
    raise RuntimeError("No location CSVs were generated.")

dfs = [pd.read_csv(f) for f in location_csvs]
df_combined = pd.concat(dfs, ignore_index=True)

combined_path = f"csv/combined_{RUN_ID}.csv"
df_combined.to_csv(combined_path, index=False, encoding="utf-8")

print(f"Combined dataset: {df_combined.shape[0]} rows x {df_combined.shape[1]} columns")
print(f"Saved: {combined_path}")
print(f"\nRows per location:")
print(df_combined["location"].value_counts().to_string())

In [ ]:
# ── Preview ───────────────────────────────────────────
print(df_combined.dtypes)
print()
df_combined.head(10)

In [ ]:
# ── Column completeness ──────────────────────────────
FINAL_COLUMNS_CHECK = [
    "location",
    "osm_id", "name", "lat", "lon", "label", "distance_m",
    "lot_area_sqft", "highway_type", "lanes",
    "dist_bus_stop_m", "dist_hospital_m", "dist_school_m", "dist_park_m",
    "res_ratio", "com_ratio",
    "median_income",
]

print("Column completeness (combined):")
for col in FINAL_COLUMNS_CHECK:
    if col in df_combined.columns:
        n = df_combined[col].notna().sum()
        pct = 100 * n / len(df_combined)
        print(f"  {col:<22s} {n:>5d}/{len(df_combined)}  ({pct:.1f}%)")
    else:
        print(f"  {col:<22s}     -  (not present)")